# 1_0_5 Gene-to-m/z Matching with GNN Integration

This notebook is the **real data analysis** step, running **after** `0_2` (weight optimization).

It modifies the original `1_0` pipeline to:
1. Load optimized weights from `optimal_weights.json` (from `0_2`)
2. Load the trained GNN model from `final_model.pth` (from `0_1_5`)
3. Use **voting fusion** (consistent with `0_2`) instead of hardcoded weighted sum
4. Add GNN cosine similarity as the 9th metric

## Pipeline: `0_1_0` → `0_1_5` → `0_2` → **`1_0_5`**

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

import numpy as np
import scanpy as sc
import json
import os
import time
import pickle
import warnings
from dataclasses import dataclass
from typing import Dict, Optional

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from scipy.interpolate import griddata
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# --- Paths ---
WEIGHTS_JSON = './optimal_weights.json'  # From 0_2
GNN_MODEL_PATH = './gnn_integration_output/final_model.pth'  # From 0_1_5

# Real data paths (modify these for your actual data)
RNA_PIXEL_SIZE = 55  # um (Visium)
MSI_PIXEL_SIZE = 60  # um
TOP_K_MATCHES = 26

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common_synced.h5ad", "halfbrain_yc_2_filtered_common_synced.h5ad",
    "halfbrain_yc_3_filtered_common_synced.h5ad", "halfbrain_yc_4_filtered_common_synced.h5ad",
    "halfbrain_yad_1_filtered_common_synced.h5ad", "halfbrain_yad_2_filtered_common_synced.h5ad",
    "halfbrain_yad_3_filtered_common_synced.h5ad", "halfbrain_yad_4_filtered_common_synced.h5ad",
    "halfbrain_ac_1_filtered_common_synced.h5ad", "halfbrain_ac_2_filtered_common_synced.h5ad",
    "halfbrain_ac_3_filtered_common_synced.h5ad", "halfbrain_ac_4_filtered_common_synced.h5ad",
    "halfbrain_aad_1_filtered_common_synced.h5ad", "halfbrain_aad_2_filtered_common_synced.h5ad",
    "halfbrain_aad_3_filtered_common_synced.h5ad", "halfbrain_aad_4_filtered_common_synced.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

AAD_TARGET_GENES = [
    'Eno1', 'Mapt', 'Thy1', 'Pmch', 'Atp1a3', 'Rac1', 'Rsrp1', 'Snhg11', 'Tubb4a',
    'Rasgrf1', 'Hsp90ab1', 'Elavl3', 'App', 'Syp', 'AC149090.1',
    'Aplp1', 'Apoe', 'Meg3', 'Gnas', 'Pkm'
]

OUTPUT_DIR = './gene_to_mz_results_with_gnn'
os.makedirs(OUTPUT_DIR, exist_ok=True)
for subdir in ['gene_visualizations', 'match_visualizations', 'descriptors']:
    os.makedirs(os.path.join(OUTPUT_DIR, subdir), exist_ok=True)

# --- Load optimized weights ---
with open(WEIGHTS_JSON, 'r') as f:
    weights_data = json.load(f)

METRIC_ROOTS = weights_data['metric_roots']
OPTIMAL_WEIGHTS = [weights_data['weights'][m] for m in METRIC_ROOTS]

print(f"\nLoaded weights from: {WEIGHTS_JSON}")
print(f"Metrics: {METRIC_ROOTS}")
print(f"Weights:")
for m, w in zip(METRIC_ROOTS, OPTIMAL_WEIGHTS):
    print(f"  {m:<25} : {w:.4f}")
print(f"\nOptimized accuracy: {weights_data['accuracy']:.2%}")

# GNN config (must match training)
GNN_CONFIG = {
    'hidden_dim': 128,
    'embedding_dim': 64,
    'projection_dim': 64,
    'num_heads': 4,
    'num_layers': 3,
    'dropout': 0.1,
    'k_neighbors': 8,
}

In [ ]:
# ============================================================================
# GNN MODEL (from 0_1_5)
# ============================================================================

class SpatialEncoder(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=128, embedding_dim=64,
                 num_heads=4, num_layers=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(
                GATv2Conv(hidden_dim, hidden_dim // num_heads,
                         heads=num_heads, dropout=dropout, concat=True)
            )
            self.norms.append(nn.LayerNorm(hidden_dim))
        self.pool_attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1)
        )
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, embedding_dim),
            nn.LayerNorm(embedding_dim)
        )

    def forward(self, x, edge_index):
        h = self.input_proj(x)
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index)
            h_new = norm(h_new)
            h_new = F.gelu(h_new)
            h = h + h_new
        attn_weights = self.pool_attn(h)
        attn_weights = F.softmax(attn_weights, dim=0)
        global_emb = (h * attn_weights).sum(dim=0)
        return self.output_proj(global_emb)


class CrossModalGNN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.rna_encoder = SpatialEncoder(
            input_dim=3, hidden_dim=config['hidden_dim'],
            embedding_dim=config['embedding_dim'], num_heads=config['num_heads'],
            num_layers=config['num_layers'], dropout=config['dropout']
        )
        self.msi_encoder = SpatialEncoder(
            input_dim=3, hidden_dim=config['hidden_dim'],
            embedding_dim=config['embedding_dim'], num_heads=config['num_heads'],
            num_layers=config['num_layers'], dropout=config['dropout']
        )
        self.rna_projection = nn.Sequential(
            nn.Linear(config['embedding_dim'], config['embedding_dim']),
            nn.GELU(),
            nn.Linear(config['embedding_dim'], config['projection_dim'])
        )
        self.msi_projection = nn.Sequential(
            nn.Linear(config['embedding_dim'], config['embedding_dim']),
            nn.GELU(),
            nn.Linear(config['embedding_dim'], config['projection_dim'])
        )

    def encode_rna(self, x, edge_index):
        emb = self.rna_encoder(x, edge_index)
        proj = self.rna_projection(emb)
        return F.normalize(proj, dim=-1)

    def encode_msi(self, x, edge_index):
        emb = self.msi_encoder(x, edge_index)
        proj = self.msi_projection(emb)
        return F.normalize(proj, dim=-1)


def build_knn_graph(coords, k=8):
    nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm='ball_tree').fit(coords)
    _, indices = nbrs.kneighbors(coords)
    edges = []
    for i in range(len(coords)):
        for j in indices[i, 1:]:
            edges.append([i, j])
            edges.append([j, i])
    return torch.tensor(edges, dtype=torch.long).t().contiguous()


def make_gnn_features(coords, values):
    values_norm = (values - values.mean()) / (values.std() + 1e-8)
    x_norm = (coords[:, 0] - coords[:, 0].mean()) / (coords[:, 0].std() + 1e-8)
    y_norm = (coords[:, 1] - coords[:, 1].mean()) / (coords[:, 1].std() + 1e-8)
    return torch.tensor(np.column_stack([values_norm, x_norm, y_norm]), dtype=torch.float32)


# Load GNN model
gnn_model = CrossModalGNN(GNN_CONFIG).to(DEVICE)
gnn_model.load_state_dict(torch.load(GNN_MODEL_PATH, map_location=DEVICE))
gnn_model.eval()
print(f"GNN model loaded from: {GNN_MODEL_PATH}")

In [ ]:
# ============================================================================
# TRADITIONAL SPATIAL MATCHING FUNCTIONS (from 1_0)
# ============================================================================

def rotate_180(coords):
    center = coords.mean(axis=0)
    return 2 * center - coords


def align_rna_to_msi(rna_coords, msi_coords):
    rotated = rotate_180(rna_coords)
    rna_min, rna_max = rotated.min(axis=0), rotated.max(axis=0)
    msi_min, msi_max = msi_coords.min(axis=0), msi_coords.max(axis=0)
    rna_range = rna_max - rna_min
    msi_range = msi_max - msi_min
    scale = msi_range / (rna_range + 1e-8)
    return (rotated - rna_min) * scale + msi_min


def compute_value_histogram(values, n_bins=50):
    hist, _ = np.histogram(values, bins=n_bins, density=True)
    return hist / (hist.sum() + 1e-8)


def compute_spatial_histogram(coords, values, n_bins=10):
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    x_bins = np.clip((norm[:, 0] * n_bins).astype(int), 0, n_bins - 1)
    y_bins = np.clip((norm[:, 1] * n_bins).astype(int), 0, n_bins - 1)
    flat_idx = y_bins * n_bins + x_bins
    hist = np.bincount(flat_idx, weights=values, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    counts = np.bincount(flat_idx, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    hist = np.divide(hist, counts, where=counts > 0, out=np.zeros_like(hist))
    hist_min, hist_max = hist.min(), hist.max()
    if hist_max > hist_min:
        hist = (hist - hist_min) / (hist_max - hist_min)
    return hist


def compute_radial_profile(coords, values, n_rings=10):
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    centroid = norm.mean(axis=0)
    distances = np.linalg.norm(norm - centroid, axis=1)
    max_dist = distances.max() + 1e-8
    ring_idx = np.clip((distances / max_dist * n_rings).astype(int), 0, n_rings - 1)
    profile = np.bincount(ring_idx, weights=values, minlength=n_rings)
    counts = np.bincount(ring_idx, minlength=n_rings)
    profile = np.divide(profile, counts, where=counts > 0, out=np.zeros_like(profile, dtype=float))
    prof_min, prof_max = profile.min(), profile.max()
    if prof_max > prof_min:
        profile = (profile - prof_min) / (prof_max - prof_min)
    return profile


def compute_quadrant_stats(coords, values, n_div=3):
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    x_bins = np.clip((norm[:, 0] * n_div).astype(int), 0, n_div - 1)
    y_bins = np.clip((norm[:, 1] * n_div).astype(int), 0, n_div - 1)
    flat_idx = y_bins * n_div + x_bins
    stats = np.zeros(n_div * n_div * 2)
    for q in range(n_div * n_div):
        mask = flat_idx == q
        if mask.sum() > 0:
            q_vals = values[mask]
            stats[q * 2] = np.mean(q_vals)
            stats[q * 2 + 1] = np.std(q_vals)
    stats_min, stats_max = stats.min(), stats.max()
    if stats_max > stats_min:
        stats = (stats - stats_min) / (stats_max - stats_min)
    return stats


def compute_morans_i_vectorized(coords, values, indices):
    n = len(values)
    mean_val = values.mean()
    deviations = values - mean_val
    denom = np.sum(deviations ** 2)
    if denom == 0:
        return 0.0
    neighbor_deviations = deviations[indices[:, 1:]]
    numer = np.sum(deviations[:, np.newaxis] * neighbor_deviations)
    w_sum = indices.shape[0] * (indices.shape[1] - 1)
    return (n / w_sum) * (numer / denom) if w_sum > 0 else 0.0


@dataclass
class SpatialSignature:
    sample_id: str
    feature_name: str
    feature_type: str
    node_importance: np.ndarray
    value_histogram: np.ndarray = None
    spatial_histogram: np.ndarray = None
    radial_profile: np.ndarray = None
    quadrant_stats: np.ndarray = None
    morans_i: float = 0.0
    coordinates: np.ndarray = None
    raw_values: np.ndarray = None
    aligned_coordinates: Optional[np.ndarray] = None


def compute_coordinate_based_similarity(gene_sig, mz_sig, grid_res=50):
    gene_coords = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None else gene_sig.coordinates
    mz_coords = mz_sig.coordinates
    x_min = min(gene_coords[:, 0].min(), mz_coords[:, 0].min())
    x_max = max(gene_coords[:, 0].max(), mz_coords[:, 0].max())
    y_min = min(gene_coords[:, 1].min(), mz_coords[:, 1].min())
    y_max = max(gene_coords[:, 1].max(), mz_coords[:, 1].max())
    grid_x, grid_y = np.meshgrid(np.linspace(x_min, x_max, grid_res), np.linspace(y_min, y_max, grid_res))
    gene_grid = griddata(gene_coords, gene_sig.raw_values, (grid_x, grid_y), method='linear')
    mz_grid = griddata(mz_coords, mz_sig.raw_values, (grid_x, grid_y), method='linear')
    gene_imp_grid = griddata(gene_coords, gene_sig.node_importance, (grid_x, grid_y), method='linear')
    mz_imp_grid = griddata(mz_coords, mz_sig.node_importance, (grid_x, grid_y), method='linear')
    mask = ~(np.isnan(gene_grid) | np.isnan(mz_grid))
    value_corr = 0
    if mask.sum() > 10:
        r, _ = pearsonr(gene_grid[mask], mz_grid[mask])
        value_corr = r if not np.isnan(r) else 0
    mask_imp = ~(np.isnan(gene_imp_grid) | np.isnan(mz_imp_grid))
    importance_iou, imp_corr = 0, 0
    if mask_imp.sum() > 0:
        g_imp = gene_imp_grid.copy()
        m_imp = mz_imp_grid.copy()
        g_imp[np.isnan(g_imp)] = 0
        m_imp[np.isnan(m_imp)] = 0
        g_imp = g_imp / (g_imp.max() + 1e-8)
        m_imp = m_imp / (m_imp.max() + 1e-8)
        importance_iou = np.minimum(g_imp, m_imp).sum() / (np.maximum(g_imp, m_imp).sum() + 1e-8)
        r, _ = pearsonr(g_imp[mask_imp], m_imp[mask_imp])
        imp_corr = r if not np.isnan(r) else 0
    return {'value_correlation': value_corr, 'importance_iou': importance_iou, 'importance_correlation': imp_corr}


def compute_descriptor_similarity(gene_sig, mz_sig):
    def safe_pearsonr(a, b):
        if a.std() > 0 and b.std() > 0:
            r, _ = pearsonr(a, b)
            return r if not np.isnan(r) else 0
        return 0
    val_hist_corr = safe_pearsonr(gene_sig.value_histogram, mz_sig.value_histogram)
    spatial_hist_corr = safe_pearsonr(gene_sig.spatial_histogram.flatten(), mz_sig.spatial_histogram.flatten())
    radial_corr = safe_pearsonr(gene_sig.radial_profile, mz_sig.radial_profile)
    quad_corr = safe_pearsonr(gene_sig.quadrant_stats, mz_sig.quadrant_stats)
    morans_sim = 1 / (1 + abs(gene_sig.morans_i - mz_sig.morans_i))
    return {'value_hist_corr': val_hist_corr, 'spatial_hist_corr': spatial_hist_corr,
            'radial_corr': radial_corr, 'quadrant_corr': quad_corr, 'morans_similarity': morans_sim}


print("Traditional matching functions defined.")

In [ ]:
# ============================================================================
# VOTING FUSION (consistent with 0_2)
# ============================================================================

# Mapping from METRIC_ROOTS names to actual score dict keys
# The traditional metrics come from compute_coordinate_based_similarity / compute_descriptor_similarity
METRIC_KEY_MAP = {
    'Value_Correlation': 'value_correlation',
    'Importance_IoU': 'importance_iou',
    'Importance_Correlation': 'importance_correlation',
    'Value_Hist_Corr': 'value_hist_corr',
    'Spatial_Hist_Corr': 'spatial_hist_corr',
    'Radial_Corr': 'radial_corr',
    'Quadrant_Corr': 'quadrant_corr',
    'Morans_Sim': 'morans_similarity',
    'GNN_Cosine': 'gnn_cosine',  # Special: handled separately
}


def compute_all_metric_scores(gene_sig, mz_sigs, gnn_gene_emb, gnn_mz_embeddings):
    """
    For each MZ candidate, compute all 9 metric scores.
    
    Returns dict: mz_name -> {metric_name: (best_mz, score)} per metric
    Actually returns per-metric best: {metric_name: (best_mz_name, best_score)}
    """
    # For each metric, find the best MZ and its score
    metric_bests = {}  # metric_name -> (best_mz, best_score)
    
    # Initialize: compute scores for all MZ candidates across all metrics
    all_scores = {}  # mz_name -> {metric_key: score}
    
    for mz_name, mz_sig in mz_sigs.items():
        coord_sim = compute_coordinate_based_similarity(gene_sig, mz_sig)
        desc_sim = compute_descriptor_similarity(gene_sig, mz_sig)
        
        scores = {}
        scores.update(coord_sim)
        scores.update(desc_sim)
        
        # Add GNN cosine score
        if mz_name in gnn_mz_embeddings:
            scores['gnn_cosine'] = float(np.dot(gnn_gene_emb, gnn_mz_embeddings[mz_name]))
        else:
            scores['gnn_cosine'] = 0.0
        
        all_scores[mz_name] = scores
    
    # For each metric, find the best MZ
    for metric_root in METRIC_ROOTS:
        metric_key = METRIC_KEY_MAP[metric_root]
        best_mz = None
        best_score = -float('inf')
        for mz_name, scores in all_scores.items():
            s = scores.get(metric_key, 0.0)
            if s > best_score:
                best_score = s
                best_mz = mz_name
        metric_bests[metric_root] = (best_mz, best_score)
    
    return metric_bests, all_scores


def voting_fusion(metric_bests, weights):
    """
    Voting fusion matching 0_2's logic.
    Each metric nominates its best MZ + score, weighted vote.
    Returns the winning MZ.
    """
    votes = {}  # mz_name -> weighted score sum
    for i, metric_root in enumerate(METRIC_ROOTS):
        best_mz, best_score = metric_bests[metric_root]
        if best_mz is None:
            continue
        if best_mz not in votes:
            votes[best_mz] = 0
        votes[best_mz] += best_score * weights[i]
    
    if not votes:
        return None
    return max(votes, key=votes.get)


print("Voting fusion defined.")

In [ ]:
# ============================================================================
# MAIN ANALYSIS
# ============================================================================

print("="*70)
print("GENE-TO-M/Z MATCHING WITH GNN INTEGRATION")
print(f"RNA: {RNA_PIXEL_SIZE}um | MSI: {MSI_PIXEL_SIZE}um")
print(f"Metrics: {len(METRIC_ROOTS)} (8 traditional + GNN_Cosine)")
print("="*70)

# --- Load data ---
print("\nLoading data...")
rna_data = {}
for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
    path = os.path.join(RNA_INPUT_FOLDER, file)
    if os.path.exists(path):
        rna_data[sample_id] = sc.read_h5ad(path)
        print(f"  RNA {sample_id}: {rna_data[sample_id].shape}")

msi_data = {}
for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
    path = os.path.join(MSI_INPUT_FOLDER, file)
    if os.path.exists(path):
        msi_data[sample_id] = sc.read_h5ad(path)
        print(f"  MSI {sample_id}: {msi_data[sample_id].shape}")

# NN cache
_nn_cache = {}
def get_nn_indices(coords, k, cache_key):
    full_key = f"{cache_key}_{k}"
    if full_key not in _nn_cache:
        coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
        norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
        k_actual = min(k + 1, len(coords))
        nn = NearestNeighbors(n_neighbors=k_actual)
        nn.fit(norm)
        _, indices = nn.kneighbors(norm)
        _nn_cache[full_key] = indices
    return _nn_cache[full_key]

def compute_bio_importance(coords, values, k, nn_indices):
    neighbor_vals = values[nn_indices[:, 1:]]
    local_var = np.var(neighbor_vals, axis=1)
    lv_min, lv_max = local_var.min(), local_var.max()
    if lv_max > lv_min:
        local_var = (local_var - lv_min) / (lv_max - lv_min)
    else:
        local_var = np.zeros_like(local_var)
    v_min, v_max = values.min(), values.max()
    if v_max > v_min:
        val_norm = (values - v_min) / (v_max - v_min)
    else:
        val_norm = np.zeros_like(values)
    return 0.5 * local_var + 0.5 * val_norm

def extract_signature(coords, values, sample_id, feature_name, feature_type,
                      n_neighbors, nn_indices, aligned_coords=None):
    bio_imp = compute_bio_importance(coords, values, n_neighbors, nn_indices)
    return SpatialSignature(
        sample_id=sample_id, feature_name=feature_name, feature_type=feature_type,
        node_importance=bio_imp, value_histogram=compute_value_histogram(values),
        spatial_histogram=compute_spatial_histogram(coords, values),
        radial_profile=compute_radial_profile(coords, values),
        quadrant_stats=compute_quadrant_stats(coords, values),
        morans_i=compute_morans_i_vectorized(coords, values, nn_indices),
        coordinates=coords, raw_values=values, aligned_coordinates=aligned_coords)

# --- Run analysis ---
all_results = []

gene_avail = {gene: {s: gene in rna_data[s].var_names
                     for s in RNA_SAMPLE_IDS if s in rna_data}
              for gene in AAD_TARGET_GENES}

print("\nGene availability:")
for gene in AAD_TARGET_GENES:
    n = sum(gene_avail[gene].values())
    print(f"  {gene}: {n}/{len(RNA_SAMPLE_IDS)}")

for gene in AAD_TARGET_GENES:
    print(f"\n{'='*50}")
    print(f"Gene: {gene}")
    print(f"{'='*50}")
    
    available = [s for s, a in gene_avail[gene].items() if a]
    if not available:
        continue
    
    for rna_sample in available:
        t0 = time.time()
        print(f"\n  {rna_sample}", flush=True)
        adata = rna_data[rna_sample]
        rna_coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
        gene_idx = list(adata.var_names).index(gene)
        gene_expr = adata.X[:, gene_idx].toarray().flatten() if hasattr(adata.X, 'toarray') else adata.X[:, gene_idx].flatten()
        
        msi_sample = rna_sample
        if msi_sample not in msi_data:
            print(f"    MSI {msi_sample} not found")
            continue
        
        msi_adata = msi_data[msi_sample]
        msi_coords = np.column_stack([msi_adata.obs['x_um'].values, msi_adata.obs['y_um'].values])
        aligned_coords = align_rna_to_msi(rna_coords, msi_coords)
        
        # Traditional signatures
        rna_nn_indices = get_nn_indices(rna_coords, 6, f"rna_{rna_sample}")
        msi_nn_indices = get_nn_indices(msi_coords, 8, f"msi_{msi_sample}")
        
        gene_sig = extract_signature(
            rna_coords, gene_expr, rna_sample, gene, 'gene', 6, rna_nn_indices, aligned_coords
        )
        
        # Extract MSI signatures (traditional)
        print(f"    Extracting MSI signatures...", flush=True)
        msi_matrix = msi_adata.X.toarray() if hasattr(msi_adata.X, 'toarray') else msi_adata.X
        mz_names = list(msi_adata.var_names)
        
        def extract_single_mz(i):
            return mz_names[i], extract_signature(
                msi_coords, msi_matrix[:, i], msi_sample, mz_names[i], 'mz', 8, msi_nn_indices
            )
        
        mz_results = Parallel(n_jobs=-1, prefer='threads')(
            delayed(extract_single_mz)(i) for i in range(len(mz_names))
        )
        mz_sigs = dict(mz_results)
        print(f"      {len(mz_names)} m/z features processed", flush=True)
        
        # GNN embeddings
        print(f"    Computing GNN embeddings...", flush=True)
        with torch.no_grad():
            # Gene embedding
            rna_feat = make_gnn_features(rna_coords, gene_expr)
            rna_edge = build_knn_graph(rna_coords, GNN_CONFIG['k_neighbors'])
            rna_graph = Data(x=rna_feat, edge_index=rna_edge).to(DEVICE)
            gnn_gene_emb = gnn_model.encode_rna(rna_graph.x, rna_graph.edge_index).cpu().numpy()
            
            # MSI embeddings for all m/z
            gnn_mz_embeddings = {}
            msi_edge = build_knn_graph(msi_coords, GNN_CONFIG['k_neighbors'])
            for i, mz_name in enumerate(mz_names):
                msi_feat = make_gnn_features(msi_coords, msi_matrix[:, i])
                msi_graph = Data(x=msi_feat, edge_index=msi_edge).to(DEVICE)
                gnn_mz_embeddings[mz_name] = gnn_model.encode_msi(
                    msi_graph.x, msi_graph.edge_index
                ).cpu().numpy()
        print(f"      GNN: {len(gnn_mz_embeddings)} embeddings computed", flush=True)
        
        # Compute all metrics and voting fusion
        print(f"    Computing metrics & voting...", flush=True)
        metric_bests, all_scores = compute_all_metric_scores(
            gene_sig, mz_sigs, gnn_gene_emb, gnn_mz_embeddings
        )
        
        # Voting fusion with optimized weights
        combined_prediction = voting_fusion(metric_bests, OPTIMAL_WEIGHTS)
        
        # Also get top-K by combined score (for detailed results)
        # Sort MZ by their total weighted vote contribution
        mz_vote_scores = {}
        for mz_name in mz_names:
            mz_vote_scores[mz_name] = 0
            for i, metric_root in enumerate(METRIC_ROOTS):
                metric_key = METRIC_KEY_MAP[metric_root]
                score = all_scores.get(mz_name, {}).get(metric_key, 0.0)
                mz_vote_scores[mz_name] += score * OPTIMAL_WEIGHTS[i]
        
        ranked_mz = sorted(mz_vote_scores.items(), key=lambda x: x[1], reverse=True)
        
        # Store results
        for rank_idx, (mz_name, combined_score) in enumerate(ranked_mz[:TOP_K_MATCHES]):
            result_row = {
                'gene': gene,
                'rna_sample': rna_sample,
                'mz_feature': mz_name,
                'msi_sample': msi_sample,
                'rank': rank_idx + 1,
                'combined_score': combined_score,
                'combined_prediction': combined_prediction,
            }
            # Add per-metric scores
            if mz_name in all_scores:
                for metric_root in METRIC_ROOTS:
                    metric_key = METRIC_KEY_MAP[metric_root]
                    result_row[metric_root] = all_scores[mz_name].get(metric_key, 0.0)
            all_results.append(result_row)
        
        elapsed = time.time() - t0
        
        # Print top matches
        print(f"    Top {min(5, TOP_K_MATCHES)} matches (voting fusion):")
        for rank_idx, (mz_name, score) in enumerate(ranked_mz[:5]):
            marker = " <-- BEST" if rank_idx == 0 else ""
            print(f"      {rank_idx+1}. {mz_name}: {score:.4f}{marker}")
        
        # Per-metric best
        print(f"    Per-metric nominations:")
        for metric_root in METRIC_ROOTS:
            best_mz, best_score = metric_bests[metric_root]
            print(f"      {metric_root:<25}: {best_mz} ({best_score:.4f})")
        
        print(f"    Time: {elapsed:.1f}s", flush=True)

# --- Save results ---
if all_results:
    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values(['gene', 'rna_sample', 'rank'])
    csv_path = os.path.join(OUTPUT_DIR, 'gene_to_mz_matches_with_gnn.csv')
    results_df.to_csv(csv_path, index=False)
    print(f"\nResults saved to: {csv_path}")
    print(f"Total rows: {len(results_df)}")
    
    # Summary: top-1 match per gene per sample
    print(f"\n{'='*70}")
    print("TOP MATCHES (Voting Fusion with GNN)")
    print(f"{'='*70}")
    top1 = results_df[results_df['rank'] == 1]
    for gene in AAD_TARGET_GENES:
        gene_top = top1[top1['gene'] == gene]
        if len(gene_top) > 0:
            # Show most common top-1 match across samples
            most_common = gene_top['mz_feature'].value_counts().index[0]
            n_samples = len(gene_top)
            n_agree = gene_top['mz_feature'].value_counts().iloc[0]
            avg_score = gene_top[gene_top['mz_feature'] == most_common]['combined_score'].mean()
            print(f"  {gene:<20} -> {most_common:<30} (score: {avg_score:.3f}, {n_agree}/{n_samples} samples agree)")

print(f"\n{'='*70}")
print("COMPLETE!")
print(f"{'='*70}")